# Fracture Caption Evaluation

Reproduces the three evaluation tables used in the paper:

1. **CLIP-based metrics** — CLIPScore, RefCLIPScore, CLIP R-Precision (Table: `clip-metrics`)
2. **Collapse-diagnostic metrics** — unique caption count / top-1 share / exact-duplicate rate per class (Table: `collapse-stats`)
3. **CIDEr-R + paired bootstrap** — per-class gap vs. the human-vs-human baseline, with Bonferroni correction (Table: `cider-bootstrap`)

Run cells top to bottom. Only the **Configuration** cell should need editing for a new prediction file.

## 0. Setup

In [ ]:
# Uncomment on a fresh environment
# %pip install torch transformers pandas numpy pillow tqdm --quiet


In [ ]:
import ast
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

pd.set_option("display.width", 120)


## 1. Configuration

Edit these for your run. `TRUE_CSV` is the human-verified ground-truth caption
file (2 caption variants/object, from `caption_fractures.py`'s output after
human correction). `PRED_CSV` is a predictions file (from `infer_mae.py` /
`infer.py`), which may have one or several sampled caption variants per object.

`CLASS_COL_*` / `CLASS_REGEX` control how the object class is recovered —
first choice is an explicit `class` column, falling back to a regex on `path`
(default matches the Breaking Bad `everyday/<class>/...` layout).

In [ ]:
# ── Files ──────────────────────────────────────────────────────────────────
TRUE_CSV   = "caption_breakingbad_test_new.csv"    # human-verified ground truth
PRED_CSV   = "predictions_test_set_v2_best.csv"    # model predictions
IMAGE_ROOT = "."                                    # root for image / images_used paths

# ── Class resolution ──────────────────────────────────────────────────────────
CLASS_COL_TRUE = "class"          # column name in TRUE_CSV, if present
CLASS_COL_PRED = "true_class"     # infer_mae.py writes this column; adjust if needed
CLASS_REGEX    = r"everyday/([^/]+)/"   # fallback: extracted from `path` if no column found

# ── Which prediction column to use where only one is needed ──────────────────
# (collapse diagnostics + CIDEr-R candidate scoring both need a single column;
#  CLIP metrics score every caption_true_*/caption_pred_* column found)
PRED_CAPTION_COL = "caption_pred_1"

# ── CLIP model ─────────────────────────────────────────────────────────────
CLIP_MODEL_NAME  = "openai/clip-vit-large-patch14"
CLIP_LOGIT_SCALE = 2.5     # CLIPScore's `w` (Hessel et al., 2021)
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
IMG_BATCH_SIZE   = 32
TXT_BATCH_SIZE   = 64

# ── CIDEr-R / bootstrap ────────────────────────────────────────────────────
CIDER_N_MAX        = 4
CIDER_SIGMA        = 6.0
N_BOOTSTRAP         = 10_000
BOOTSTRAP_SEED      = 42

OUTPUT_DIR = Path("eval_outputs_normal_best_v2")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load and merge ground-truth / predictions

Pivots both CSVs from one-row-per-(path, caption_idx) into one row per object,
with columns `caption_true_0`, `caption_true_1`, ..., `caption_pred_0`, ...
Works whether the prediction CSV has a `caption_idx` column already or just
one caption per row (in which case a `caption_idx` of 0 is assigned).

In [ ]:
def _parse_images_used(raw) -> list:
    if isinstance(raw, list):
        return raw
    if not isinstance(raw, str) or not raw.strip():
        return []
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(raw)
            if isinstance(parsed, list):
                return [str(p) for p in parsed]
        except Exception:
            continue
    return []


def load_caption_table(csv_path, prefix, class_col=None, class_out="class"):
    """Load a captions CSV and pivot to one row per `path`."""
    tbl = pd.read_csv(csv_path, sep=",")
    tbl.columns = [c.strip() for c in tbl.columns]
    if "caption_idx" not in tbl.columns:
        tbl = tbl.copy()
        tbl["caption_idx"] = tbl.groupby("path").cumcount()

    pivot = (
        tbl.pivot(index="path", columns="caption_idx", values="caption")
           .rename(columns=lambda i: f"{prefix}_{i}")
    )

    parts, names = [pivot], list(pivot.columns)
    if "images_used" in tbl.columns:
        parts.append(tbl.groupby("path")["images_used"].first())
        names.append("images_used")
    if class_col and class_col in tbl.columns:
        parts.append(tbl.groupby("path")[class_col].first())
        names.append(class_out)

    out = pd.concat(parts, axis=1)
    out.columns = names
    return out.reset_index()


true_tbl = load_caption_table(TRUE_CSV, "caption_true", class_col=CLASS_COL_TRUE, class_out="class")
pred_tbl = load_caption_table(PRED_CSV, "caption_pred", class_col=CLASS_COL_PRED, class_out="class_pred")

df = true_tbl.merge(pred_tbl, on="path", how="inner")

if "class" not in df.columns:
    df["class"] = np.nan
if "class_pred" in df.columns:
    df["class"] = df["class"].fillna(df["class_pred"])
    df = df.drop(columns=["class_pred"])
if df["class"].isna().any():
    extracted = df["path"].astype(str).str.extract(CLASS_REGEX)[0]
    df["class"] = df["class"].fillna(extracted)

true_cols = sorted(c for c in df.columns if c.startswith("caption_true_"))
pred_cols = sorted(c for c in df.columns if c.startswith("caption_pred_"))
caption_cols = true_cols + pred_cols

print(f"Merged {len(df)} objects across {df['class'].nunique()} classes.")
print(f"Reference columns: {true_cols}")
print(f"Prediction columns: {pred_cols}")
if df["class"].isna().any():
    print(f"[warn] {df['class'].isna().sum()} objects have no resolvable class.")
df.head()


## 3. CLIP-based metrics (CLIPScore, RefCLIPScore, CLIP R-Precision)

### 3.0 Load CLIP and resolve/embed images + captions once

In [ ]:
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).eval().to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)


def _unwrap(feat):
    if torch.is_tensor(feat):
        return feat
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        if hasattr(feat, attr):
            return getattr(feat, attr)
    raise TypeError(f"Expected a tensor from CLIP get_*_features, got {type(feat)}.")


def resolve_overview_image(row, image_root: Path):
    """Prefer the direct `<obj>_frac_view_overview.png` convention, else pick
    an 'overview' image out of `images_used`, else the first image that exists."""
    direct = image_root / str(row["path"]).replace(".obj", "_frac_view_overview.png")
    if direct.is_file():
        return direct
    candidates = [image_root / p for p in _parse_images_used(row.get("images_used", "[]"))]
    overview = [p for p in candidates if "overview" in p.stem.lower()]
    for p in overview + [p for p in candidates if p not in overview]:
        if p.is_file():
            return p
    return None


@torch.no_grad()
def embed_images(paths, batch_size=IMG_BATCH_SIZE):
    embeds, valid = [], []
    for start in tqdm(range(0, len(paths), batch_size), desc="Embedding images"):
        batch = paths[start:start + batch_size]
        imgs, ok = [], []
        for p in batch:
            try:
                imgs.append(Image.open(p).convert("RGB"))
                ok.append(True)
            except Exception:
                imgs.append(Image.new("RGB", (224, 224)))  # placeholder, masked out via `valid`
                ok.append(False)
        inputs = clip_processor(images=imgs, return_tensors="pt").to(DEVICE)
        feat = F.normalize(_unwrap(clip_model.get_image_features(**inputs)), dim=-1)
        embeds.append(feat.cpu())
        valid.extend(ok)
    return torch.cat(embeds, dim=0), valid


@torch.no_grad()
def embed_texts(texts, batch_size=TXT_BATCH_SIZE):
    embeds = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding texts"):
        chunk = texts[start:start + batch_size]
        chunk = [str(t) if t and not (isinstance(t, float) and pd.isna(t)) else "" for t in chunk]
        inputs = clip_processor(text=chunk, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        feat = F.normalize(_unwrap(clip_model.get_text_features(**inputs)), dim=-1)
        embeds.append(feat.cpu())
    return torch.cat(embeds, dim=0)


# ── Resolve + embed images once ───────────────────────────────────────────────
image_paths = [resolve_overview_image(row, Path(IMAGE_ROOT)) for _, row in df.iterrows()]
valid_mask  = [p is not None for p in image_paths]
n_missing   = sum(1 for v in valid_mask if not v)
if n_missing:
    print(f"[warn] {n_missing}/{len(df)} rows have no resolvable overview image "
          f"— excluded from all CLIP-based metrics below.")

eval_df    = df.loc[valid_mask].reset_index(drop=True)
eval_paths = [p for p in image_paths if p is not None]

img_embeds, _ = embed_images(eval_paths)
img_embeds = img_embeds.to(DEVICE)

text_embeds = {c: embed_texts(eval_df[c].tolist()).to(DEVICE) for c in caption_cols}
print(f"CLIP metrics will run over {len(eval_df)} objects.")


### 3.1 CLIPScore

In [ ]:
def clip_score_from_embeds(img, txt, w=CLIP_LOGIT_SCALE):
    cos = F.cosine_similarity(img, txt)
    return (w * cos.clamp(min=0)).cpu().numpy()


clipscore_summary = {}
for c in caption_cols:
    scores = clip_score_from_embeds(img_embeds, text_embeds[c])
    eval_df[f"{c}_clipscore"] = scores
    clipscore_summary[c] = {"mean": scores.mean(), "std": scores.std(), "n": len(scores)}

clipscore_table = pd.DataFrame(clipscore_summary).T
clipscore_table


### 3.2 RefCLIPScore

`RefCLIPScore(c, R, v) = harmonic_mean(CLIPScore(c, v), max_{r in R} cos(c, r))`.
Predictions are scored against the ground-truth references (`true_cols`); the
human-vs-human baseline scores each ground-truth caption against the *other*
ground-truth reference(s) (leave-one-out).

In [ ]:
def harmonic_mean(a, b, eps=1e-8):
    a = np.clip(a, 0, None)
    b = np.clip(b, 0, None)
    out = np.zeros_like(a)
    nz = (a > 0) & (b > 0)
    out[nz] = 2 * a[nz] * b[nz] / (a[nz] + b[nz] + eps)
    return out


def ref_clip_score_batch(cand_col, ref_cols):
    cand = text_embeds[cand_col]
    clip_s = clip_score_from_embeds(img_embeds, cand)
    ref_sims = torch.stack(
        [F.cosine_similarity(cand, text_embeds[r]).clamp(min=0) for r in ref_cols], dim=1
    )
    ref_text_sim = ref_sims.max(dim=1).values.cpu().numpy()
    return clip_s, ref_text_sim, harmonic_mean(clip_s, ref_text_sim)


refclip_rows = {}
for c in pred_cols:
    clip_s, reftext, refclip = ref_clip_score_batch(c, true_cols)
    eval_df[f"{c}_refclip"] = refclip
    refclip_rows[c] = {"clip_score": clip_s.mean(), "ref_text": reftext.mean(), "ref_clip_score": refclip.mean()}
refclip_table = pd.DataFrame(refclip_rows).T

baseline_table = None
if len(true_cols) >= 2:
    baseline_rows = {}
    for c in true_cols:
        others = [t for t in true_cols if t != c]
        clip_s, reftext, refclip = ref_clip_score_batch(c, others)
        baseline_rows[c] = {"clip_score": clip_s.mean(), "ref_text": reftext.mean(), "ref_clip_score": refclip.mean()}
    baseline_table = pd.DataFrame(baseline_rows).T
    print("Human-vs-human baseline (ground truth scored against the other reference):")
    display(baseline_table)

print("\nPredicted captions scored against ground-truth references:")
refclip_table


### 3.3 CLIP R-Precision

Top-1 image→text retrieval accuracy: for each image, rank every caption in the
pool by cosine similarity and check whether the caption belonging to that
image comes out on top. `pool="all"` retrieves against the whole eval set;
`pool="class"` restricts the pool to the object's own class (a harder,
collapse-sensitive setting, since templated same-class captions collide
directly with each other).

Reported as a plain proportion in `[0, 1]` (this fixes a scaling bug in an
earlier version of this notebook, where the `pool="class"` branch was
multiplied by 100 while `pool="all"` was not).

In [ ]:
def r_precision(img_emb, txt_emb, k=1):
    sim = img_emb @ txt_emb.T
    n = sim.shape[0]
    topk = sim.topk(min(k, n), dim=1).indices
    correct = torch.arange(n, device=sim.device).unsqueeze(1)
    hits = (topk == correct).any(dim=1)
    return hits.float().mean().item(), n


def run_r_precision(caption_col, pool="all", k=1):
    txt = text_embeds[caption_col]
    rows = []
    if pool == "all":
        acc, n = r_precision(img_embeds, txt, k=k)
        rows.append({"class": "ALL", "r_precision": acc, "n_pool": n, "k": k})
    elif pool == "class":
        for cls in sorted(eval_df["class"].dropna().unique()):
            idx = np.where(eval_df["class"].values == cls)[0]
            if len(idx) < 2:
                continue
            idx_t = torch.tensor(idx, device=img_embeds.device)
            acc, n = r_precision(img_embeds[idx_t], txt[idx_t], k=k)
            rows.append({"class": cls, "r_precision": acc, "n_pool": n, "k": k})
    else:
        raise ValueError("pool must be 'all' or 'class'")
    return pd.DataFrame(rows)


rprec_all = {c: run_r_precision(c, pool="all").iloc[0].to_dict() for c in caption_cols}
rprec_all_table = pd.DataFrame(rprec_all).T[["r_precision", "n_pool", "k"]]
chance_baseline = 1.0 / rprec_all_table["n_pool"].iloc[0]
print(f"Chance baseline at k=1 over a pool of {int(rprec_all_table['n_pool'].iloc[0])}: {chance_baseline:.5f}")
rprec_all_table


In [ ]:
# Optional: per-class R-Precision (harder / more diagnostic setting)
rprec_class_true = run_r_precision(true_cols[0], pool="class")
rprec_class_pred = run_r_precision(PRED_CAPTION_COL, pool="class")
rprec_class = rprec_class_true.merge(rprec_class_pred, on="class", suffixes=("_true", "_pred"))
rprec_class.sort_values("r_precision_pred")


## 4. Collapse-diagnostic metrics

Unique caption count, top-1 template share, and exact-duplicate rate, per class and overall. Computed on the full merged set (no image required).

In [ ]:
def collapse_diagnostics(data, caption_col, class_col="class"):
    def _norm(s):
        return str(s).strip().lower()

    def stats_for(sub):
        caps = sub[caption_col].map(_norm)
        n = len(caps)
        if n == 0:
            return pd.Series({"N": 0, "unique_ratio": np.nan, "top1_share": np.nan, "duplicate_rate": np.nan})
        counts = caps.value_counts()
        unique_ratio = (counts == 1).sum() / n
        top1_share = counts.iloc[0] / n
        duplicate_rate = 1.0 - unique_ratio
        return pd.Series({"N": n, "unique_ratio": unique_ratio,
                           "top1_share": top1_share, "duplicate_rate": duplicate_rate})

    per_class = (
        data.groupby(class_col, group_keys=True)
            .apply(stats_for)
            .reset_index()
            .rename(columns={class_col: "class"})
    )
    overall = stats_for(data)
    overall["class"] = "ALL"
    table = pd.concat([per_class, overall.to_frame().T], ignore_index=True)
    table["N"] = table["N"].astype(int)
    return table.sort_values("top1_share", ascending=False).reset_index(drop=True)


collapse_table = collapse_diagnostics(df, PRED_CAPTION_COL)
collapse_table


## 5. CIDEr-R + paired bootstrap vs. human-vs-human baseline

A from-scratch implementation of the mechanism CIDEr-R adds on top of
CIDEr-D: corpus-IDF-weighted n-gram cosine similarity (n=1..4) with the
standard length-Gaussian penalty, multiplied by an explicit **repetition
penalty** — the fraction of an n-gram order's occurrences that are lexically
distinct, averaged over n. A caption that repeats the same phrase scores a
lower penalty. This reproduces the mechanism described in Santos et al.
(2021), not a byte-for-byte port of their reference code — if exact
published numbers must match elsewhere, cross-check against an official
implementation.

IDF statistics are computed once over the ground-truth reference corpus
(`true_cols`), then reused for every scored caption.

In [ ]:
def tokenize(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    text = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    return text.split()


def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def compute_document_frequency(reference_lists, n_max=CIDER_N_MAX):
    """reference_lists: list of list[str] — the reference captions for each object."""
    dfreq = {n: defaultdict(int) for n in range(1, n_max + 1)}
    for refs in reference_lists:
        seen = {n: set() for n in range(1, n_max + 1)}
        for ref in refs:
            toks = tokenize(ref)
            for n in range(1, n_max + 1):
                seen[n].update(get_ngrams(toks, n))
        for n in range(1, n_max + 1):
            for ng in seen[n]:
                dfreq[n][ng] += 1
    return dfreq


def tfidf_vector(tokens, n, dfreq, n_docs):
    counts = Counter(get_ngrams(tokens, n))
    total = sum(counts.values())
    vec = {}
    for ng, c in counts.items():
        idf = np.log(max(1.0, n_docs / max(1, dfreq[n].get(ng, 0))))
        vec[ng] = (c / total if total else 0.0) * idf
    return vec


def cosine_clipped(cand_vec, ref_vec):
    """CIDEr-D numerator: clip candidate counts to the matching reference count."""
    num = sum(min(v, ref_vec.get(g, 0.0)) * ref_vec.get(g, 0.0) for g, v in cand_vec.items())
    cand_norm = np.sqrt(sum(v * v for v in cand_vec.values()))
    ref_norm = np.sqrt(sum(v * v for v in ref_vec.values()))
    if cand_norm == 0 or ref_norm == 0:
        return 0.0
    return num / (cand_norm * ref_norm)


def repetition_penalty(tokens, n_max=CIDER_N_MAX):
    fracs = []
    for n in range(1, n_max + 1):
        ngrams = get_ngrams(tokens, n)
        if ngrams:
            fracs.append(len(set(ngrams)) / len(ngrams))
    return float(np.mean(fracs)) if fracs else 1.0


def cider_r_score(candidate, references, dfreq, n_docs, n_max=CIDER_N_MAX, sigma=CIDER_SIGMA):
    cand_tokens = tokenize(candidate)
    ref_token_lists = [tokenize(r) for r in references
                        if r and not (isinstance(r, float) and pd.isna(r))]
    if not cand_tokens or not ref_token_lists:
        return 0.0

    n_scores = []
    for n in range(1, n_max + 1):
        cand_vec = tfidf_vector(cand_tokens, n, dfreq, n_docs)
        per_ref = []
        for ref_tokens in ref_token_lists:
            ref_vec = tfidf_vector(ref_tokens, n, dfreq, n_docs)
            sim = cosine_clipped(cand_vec, ref_vec)
            length_penalty = np.exp(-((len(cand_tokens) - len(ref_tokens)) ** 2) / (2 * sigma ** 2))
            per_ref.append(sim * length_penalty)
        n_scores.append(10.0 * np.mean(per_ref))

    cider_d = float(np.mean(n_scores))
    return cider_d * repetition_penalty(cand_tokens, n_max=n_max)


reference_lists = df[true_cols].values.tolist()
dfreq = compute_document_frequency(reference_lists, n_max=CIDER_N_MAX)
n_docs = len(df)


def score_column_vs_references(cand_col, ref_cols):
    out = np.empty(len(df))
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"CIDEr-R: {cand_col}")):
        out[i] = cider_r_score(row[cand_col], [row[c] for c in ref_cols], dfreq, n_docs)
    return out


pred_scores = score_column_vs_references(PRED_CAPTION_COL, true_cols)

if len(true_cols) >= 2:
    baseline_scores = score_column_vs_references(true_cols[0], [true_cols[1]])
else:
    raise ValueError("Need at least 2 ground-truth reference columns for the human-vs-human baseline.")

print(f"Mean CIDEr-R — predicted: {pred_scores.mean():.4f}  human-vs-human baseline: {baseline_scores.mean():.4f}")


### 5.1 Paired bootstrap CI on the gap (predicted − baseline), per class, Bonferroni-corrected

In [ ]:
def bootstrap_diff_samples(diffs, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    n = len(diffs)
    idx = rng.integers(0, n, size=(n_boot, n))
    return diffs[idx].mean(axis=1)


classes = sorted(df["class"].dropna().unique())
n_classes = len(classes)
alpha_bonf = 0.05 / n_classes

rows = []
for cls in classes:
    idx = np.where(df["class"].values == cls)[0]
    diffs = pred_scores[idx] - baseline_scores[idx]
    boot = bootstrap_diff_samples(diffs)
    lo95, hi95 = np.percentile(boot, [2.5, 97.5])
    lo_bonf, hi_bonf = np.percentile(boot, [100 * alpha_bonf / 2, 100 * (1 - alpha_bonf / 2)])
    rows.append({
        "class": cls, "N": len(idx), "gap": diffs.mean(),
        "ci95_lo": lo95, "ci95_hi": hi95,
        "bonferroni_significant": not (lo_bonf <= 0 <= hi_bonf),
    })

diffs_all = pred_scores - baseline_scores
boot_all = bootstrap_diff_samples(diffs_all)
lo_all, hi_all = np.percentile(boot_all, [2.5, 97.5])
rows.append({"class": "ALL", "N": len(diffs_all), "gap": diffs_all.mean(),
             "ci95_lo": lo_all, "ci95_hi": hi_all, "bonferroni_significant": None})

cider_bootstrap_table = pd.DataFrame(rows).sort_values("gap", ascending=False).reset_index(drop=True)
print(f"Bonferroni-adjusted alpha across {n_classes} classes: {alpha_bonf:.5f}")
cider_bootstrap_table


## 6. Save all metric tables

In [ ]:
clipscore_table.to_csv(OUTPUT_DIR / "clipscore_summary.csv")
refclip_table.to_csv(OUTPUT_DIR / "refclip_summary.csv")
if baseline_table is not None:
    baseline_table.to_csv(OUTPUT_DIR / "refclip_human_baseline.csv")
rprec_all_table.to_csv(OUTPUT_DIR / "clip_r_precision_all.csv")
rprec_class.to_csv(OUTPUT_DIR / "clip_r_precision_per_class.csv", index=False)
collapse_table.to_csv(OUTPUT_DIR / "collapse_diagnostics.csv", index=False)
cider_bootstrap_table.to_csv(OUTPUT_DIR / "cider_r_bootstrap.csv", index=False)

print(f"Saved all metric tables to: {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", f.name)


## 7. (Optional) Qualitative spot-check

Renders the images for one object alongside its ground-truth and predicted
captions — useful for sanity-checking a class or gap flagged above.

In [ ]:
import random
import matplotlib.pyplot as plt


def show_row(data, path_idx, image_root=IMAGE_ROOT, caption_cols=None):
    row = data.iloc[path_idx]
    caption_cols = caption_cols or true_cols + pred_cols
    images = _parse_images_used(row.get("images_used", "[]"))
    n = max(len(images), 1)
    cols = min(3, n)
    rows_ = -(-n // cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(4 * cols, 4 * rows_))
    axes = np.atleast_1d(axes).flatten()
    for i, ax in enumerate(axes):
        ax.set_xticks([]); ax.set_yticks([])
        if i < len(images):
            try:
                ax.imshow(Image.open(Path(image_root) / images[i]).convert("RGB"))
                ax.set_title(Path(images[i]).name, fontsize=8)
            except Exception as e:
                ax.text(0.5, 0.5, f"[load failed]\n{e}", ha="center", va="center", fontsize=8)
        else:
            ax.axis("off")
    fig.suptitle(f"{row.get('path','')}  (class={row.get('class','?')})", fontsize=10)
    plt.tight_layout()
    plt.show()
    for c in caption_cols:
        print(f"{c}: {row.get(c, '[missing]')}")


show_row(df, random.randrange(len(df)))
